# Notebook 3 — Categorical Feature Engineering
Real-world categorical columns from `telecom_customers.csv`: `gender`, `contract`,
`payment_method`, `internet_service`, `multiple_lines`.

In [ ]:
import pandas as pd
import numpy as np

customers = pd.read_csv("./telecom_customers.csv")
customers[["gender","contract","payment_method","internet_service","multiple_lines"]].nunique()

## 1. Types of Categorical Variables

- **Nominal**: no inherent order — e.g. `payment_method` (Electronic check, Mailed
  check, ...). One-Hot Encoding is the natural fit.
- **Ordinal**: inherent order — e.g. `contract` (Month-to-month < One year < Two year,
  in terms of commitment length). Ordinal Encoding preserves that order for the model.
- **Binary**: exactly two categories — e.g. `phone_service` (Yes/No). Can be mapped
  directly to 0/1.

## 2. One-Hot Encoding

**When to use:** low-to-moderate cardinality nominal variables, especially for linear
models / neural nets that can't infer an implicit order from integer codes.

In [ ]:
ohe = pd.get_dummies(customers["payment_method"], prefix="payment")
ohe.head()

## 3. Ordinal Encoding

**When to use:** when categories have a genuine, business-meaningful rank. Encoding
`contract` this way lets tree-based models split naturally on "commitment level"
instead of treating each category as unrelated.

In [ ]:
contract_order = {"Month-to-month": 0, "One year": 1, "Two year": 2}
customers["contract_ordinal"] = customers["contract"].map(contract_order)
customers[["contract","contract_ordinal"]].drop_duplicates()

## 4. Label Encoding

Similar mechanics to ordinal encoding but applied to variables with **no true order**
— it should only be used for tree-based models (which split on thresholds, not
magnitude), never for linear/distance-based models, because it silently implies a
fake numeric order.

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
customers["internet_service_label"] = le.fit_transform(customers["internet_service"])
dict(zip(le.classes_, le.transform(le.classes_)))

## 5. Frequency Encoding

**When to use:** high-cardinality nominal columns where one-hot would explode
dimensionality, and category *popularity itself* may be predictive (rarer payment
methods, for instance, are used more often by less "typical" customers).

In [ ]:
freq_map = customers["payment_method"].value_counts(normalize=True)
customers["payment_method_freq"] = customers["payment_method"].map(freq_map)
customers[["payment_method","payment_method_freq"]].drop_duplicates()

## 6. Target Encoding

**When to use:** high-cardinality categoricals where you want to inject the actual
relationship with the target — powerful, but **high leakage risk** if not done inside
a proper cross-validated pipeline (a category's target encoding must never be computed
using its own row's target value).

In [ ]:
# Correct version: fit target encoding on a TRAIN split only, using K-Fold to avoid leakage
from sklearn.model_selection import KFold

df = customers.copy()
df["churn_binary"] = (df["churn"] == "Yes").astype(int)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
df["payment_method_target_enc"] = np.nan

for train_idx, val_idx in kf.split(df):
    fold_means = df.iloc[train_idx].groupby("payment_method")["churn_binary"].mean()
    df.loc[df.index[val_idx], "payment_method_target_enc"] = (
        df.iloc[val_idx]["payment_method"].map(fold_means)
    )

df[["payment_method","churn_binary","payment_method_target_enc"]].groupby("payment_method").mean(numeric_only=True)

> **Why K-Fold here matters:** if we had instead done
> `df.groupby("payment_method")["churn_binary"].transform("mean")` directly on the full
> dataset, every row's own churn outcome would leak into its own encoded value — this
> is exactly the kind of leakage covered formally in Notebook 12.

## 7. Rare Category Grouping & Cardinality Management

**When to use:** when a categorical column has many rare levels (e.g. a real ZIP-code
or free-text "device model" column) that would otherwise create one-hot columns seen
only a handful of times — these tend to overfit.

In [ ]:
# Demonstration on a synthetically higher-cardinality column
device_models = ["ModelA","ModelB","ModelC","ModelD","ModelE","ModelF","ModelG"]
rng = np.random.default_rng(1)
probs = [0.35,0.25,0.15,0.10,0.07,0.05,0.03]
customers["device_model"] = rng.choice(device_models, size=len(customers), p=probs)

counts = customers["device_model"].value_counts(normalize=True)
rare_levels = counts[counts < 0.05].index
customers["device_model_grouped"] = customers["device_model"].where(
    ~customers["device_model"].isin(rare_levels), "Other"
)
customers["device_model_grouped"].value_counts()

## 8. Handling Unknown/Unseen Categories

In production, the test/serving data can contain categories never seen in training
(e.g. a brand-new payment provider). A robust pipeline must define a fallback —
typically an explicit `"Unknown"` bucket — rather than crashing or silently
mishandling it. `OneHotEncoder(handle_unknown="ignore")` in scikit-learn is the
standard production-safe approach, shown in Notebook 13's pipeline.

## 9. Combining Categories (Feature Crosses)

**When to use:** when the *interaction* of two categorical columns is more predictive
than either alone — a classic case for churn: contract type × payment method.

In [ ]:
customers["contract_x_payment"] = customers["contract"] + " | " + customers["payment_method"]
churn_rate_by_combo = (
    customers.assign(churn_binary=(customers["churn"]=="Yes").astype(int))
    .groupby("contract_x_payment")["churn_binary"].mean()
    .sort_values(ascending=False)
)
churn_rate_by_combo.head(6)

This immediately surfaces a real, actionable business insight: **month-to-month
customers paying by electronic check churn dramatically more** than any other
combination — exactly the kind of segment a retention team would target first.

## Summary — Feature Justification

| Feature | Source | Logic | Leakage Risk | Decision |
|---|---|---|---|---|
| `contract_ordinal` | contract | mapped rank 0/1/2 | None | **Retain** |
| `payment_method_freq` | payment_method | category frequency | None | **Retain** |
| `payment_method_target_enc` | payment_method, churn | K-Fold mean target | Low (K-Fold prevents self-leakage) | **Retain**, only if computed inside CV |
| `contract_x_payment` | contract, payment_method | string concat | None | **Retain** — strong business insight |
| `device_model_grouped` | device_model | rare-category bucketing | None | **Retain** |